# Naive Bayes — Wine & Fraud Detection

This notebook applies the **Naive Bayes** classifier to two datasets:

1. **Wine** — sklearn's built-in wine recognition dataset (3-class, 13 numeric features, 178 samples).  
2. **Fraud Detection** — a local CSV with 10 000 credit-card transactions labelled as fraudulent or not.

For each dataset we will:
- Load and explore the data
- Preprocess features when necessary
- Train a Naive Bayes model
- Evaluate with accuracy, classification report, and a confusion matrix

In [ ]:
# ──────────────────────────────────────────────
# Imports shared by both experiments
# ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

---
## Part 1 — Wine Dataset

The wine dataset contains 178 samples of wines from three different cultivars,
each described by 13 continuous chemical measurements (alcohol, malic acid, ash, etc.).

Because the features are **continuous**, we use **GaussianNB**, which assumes
each feature follows a normal distribution within each class.

In [ ]:
# ──────────────────────────────────────────────
# 1.1  Load the wine dataset from sklearn
# ──────────────────────────────────────────────
from sklearn.datasets import load_wine

wine = load_wine()

# Wrap it in a DataFrame for easier exploration
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine["target"] = wine.target

print(f"Shape: {df_wine.shape}")
print(f"Classes: {wine.target_names}")
df_wine.head()

In [ ]:
# ──────────────────────────────────────────────
# 1.2  Quick look at the class distribution
# ──────────────────────────────────────────────
print("Class counts:")
print(df_wine["target"].value_counts().sort_index())

# The dataset is roughly balanced (59 / 71 / 48), so no
# special resampling is needed.

In [ ]:
# ──────────────────────────────────────────────
# 1.3  Split into train / test sets
# ──────────────────────────────────────────────
X_wine = df_wine.drop(columns="target")
y_wine = df_wine["target"]

# 80 / 20 split; stratify keeps class proportions in both sets
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

print(f"Training samples: {len(y_train_w)}")
print(f"Test samples:     {len(y_test_w)}")

In [ ]:
# ──────────────────────────────────────────────
# 1.4  Feature scaling (optional but helpful for GaussianNB)
# ──────────────────────────────────────────────
# GaussianNB estimates mean and variance per feature per class.
# Scaling doesn't change the decision boundaries, but it can
# improve numerical stability when feature ranges differ a lot.

scaler_w = StandardScaler()
X_train_w_sc = scaler_w.fit_transform(X_train_w)  # fit on train only
X_test_w_sc  = scaler_w.transform(X_test_w)        # transform test with same params

In [ ]:
# ──────────────────────────────────────────────
# 1.5  Train GaussianNB and predict
# ──────────────────────────────────────────────
gnb_wine = GaussianNB()
gnb_wine.fit(X_train_w_sc, y_train_w)

y_pred_w = gnb_wine.predict(X_test_w_sc)

print(f"Predicted: {y_pred_w}")
print(f"Actual:    {y_test_w.values}")

In [ ]:
# ──────────────────────────────────────────────
# 1.6  Evaluation — accuracy & classification report
# ──────────────────────────────────────────────
acc_w = accuracy_score(y_test_w, y_pred_w)
report_w = classification_report(
    y_test_w, y_pred_w, target_names=wine.target_names
)

print(f"Accuracy: {acc_w:.4f}")
print(f"\nClassification Report:\n{report_w}")

In [ ]:
# ──────────────────────────────────────────────
# 1.7  Confusion matrix heatmap
# ──────────────────────────────────────────────
cm_w = confusion_matrix(y_test_w, y_pred_w)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm_w,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=wine.target_names,
    yticklabels=wine.target_names,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Wine — Confusion Matrix (GaussianNB)")
plt.tight_layout()
plt.show()

---
## Part 2 — Fraud Detection Dataset

The local `fraud_detection.csv` contains 10 000 rows with the columns:

| Column | Type | Description |
|---|---|---|
| Profession | categorical | DOCTOR, LAWYER, ENGINEER |
| Income | int | Annual income |
| Credit_card_number | int | Card number (ID-like, not predictive) |
| Expiry | str | Card expiry date |
| Security_code | int | 3-digit CVV |
| Fraud | int (0/1) | Target label |

**Preprocessing notes:**
- `Credit_card_number` is essentially an ID and will be dropped.
- `Expiry` will be dropped because it is an arbitrary date string with no ordinal meaning for this model.
- `Profession` is categorical and must be encoded numerically.
- `Security_code` is kept as-is (numeric).

We again use **GaussianNB** since the remaining features are numeric.

In [ ]:
# ──────────────────────────────────────────────
# 2.1  Load the fraud detection CSV
# ──────────────────────────────────────────────
df_fraud = pd.read_csv("fraud_detection.csv")

print(f"Shape: {df_fraud.shape}")
print(f"\nFirst rows:")
df_fraud.head()

In [ ]:
# ──────────────────────────────────────────────
# 2.2  Explore class balance and basic stats
# ──────────────────────────────────────────────
print("Fraud distribution:")
print(df_fraud["Fraud"].value_counts())
print(f"\nProfession categories: {df_fraud['Profession'].unique()}")

# The dataset is almost perfectly balanced (~50/50),
# so no resampling is required.
df_fraud.describe()

In [ ]:
# ──────────────────────────────────────────────
# 2.3  Preprocessing — drop non-predictive columns,
#       encode the categorical feature
# ──────────────────────────────────────────────

# Drop columns that do not carry useful predictive signal
df_fraud_clean = df_fraud.drop(columns=["Credit_card_number", "Expiry"])

# Encode 'Profession' (DOCTOR, LAWYER, ENGINEER) as integers
le = LabelEncoder()
df_fraud_clean["Profession"] = le.fit_transform(df_fraud_clean["Profession"])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
df_fraud_clean.head()

In [ ]:
# ──────────────────────────────────────────────
# 2.4  Train / test split
# ──────────────────────────────────────────────
X_fraud = df_fraud_clean.drop(columns="Fraud")
y_fraud = df_fraud_clean["Fraud"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

print(f"Training samples: {len(y_train_f)}")
print(f"Test samples:     {len(y_test_f)}")
print(f"Fraud in test:    {y_test_f.sum()} / {len(y_test_f)}")

In [ ]:
# ──────────────────────────────────────────────
# 2.5  Scale features for numerical stability
# ──────────────────────────────────────────────
scaler_f = StandardScaler()
X_train_f_sc = scaler_f.fit_transform(X_train_f)
X_test_f_sc  = scaler_f.transform(X_test_f)

In [ ]:
# ──────────────────────────────────────────────
# 2.6  Train GaussianNB and predict
# ──────────────────────────────────────────────
gnb_fraud = GaussianNB()
gnb_fraud.fit(X_train_f_sc, y_train_f)

y_pred_f = gnb_fraud.predict(X_test_f_sc)

# Print a small sample of predictions vs actuals
print("First 20 predictions:", y_pred_f[:20])
print("First 20 actuals:    ", y_test_f.values[:20])

In [ ]:
# ──────────────────────────────────────────────
# 2.7  Evaluation — accuracy & classification report
# ──────────────────────────────────────────────
acc_f = accuracy_score(y_test_f, y_pred_f)
report_f = classification_report(
    y_test_f, y_pred_f, target_names=["Not Fraud", "Fraud"]
)

print(f"Accuracy: {acc_f:.4f}")
print(f"\nClassification Report:\n{report_f}")

In [ ]:
# ──────────────────────────────────────────────
# 2.8  Confusion matrix heatmap
# ──────────────────────────────────────────────
cm_f = confusion_matrix(y_test_f, y_pred_f)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm_f,
    annot=True,
    fmt="d",
    cmap="Oranges",
    xticklabels=["Not Fraud", "Fraud"],
    yticklabels=["Not Fraud", "Fraud"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Fraud Detection — Confusion Matrix (GaussianNB)")
plt.tight_layout()
plt.show()

---
## Summary

| Dataset | Variant | Classes | Features | Samples |
|---|---|---|---|---|
| Wine | GaussianNB | 3 (cultivars) | 13 continuous | 178 |
| Fraud Detection | GaussianNB | 2 (fraud / not) | 3 (after dropping ID-like cols) | 10 000 |

**Key takeaways:**
- Naive Bayes is fast and works well as a baseline, especially when the Gaussian (normal-distribution) assumption roughly holds.
- For the wine dataset, 13 well-measured chemical features usually give strong separation → high accuracy.
- For fraud detection, the features available (`Profession`, `Income`, `Security_code`) may carry limited discriminative power, so accuracy can be modest — this highlights that **feature quality matters more than model complexity**.